# TP2 — Padding, stride et pooling

**Cours :** Réseaux de neurones convolutifs (CNN) — 30 h — Master 1 SDIA
**Enseignante :** Dr. Noulapeu Ngaffo Armielle — ENSPD, Université de Douala
**Durée conseillée :** 1 h 30  ·  **Barème :** 10 points

---

## Objectifs

- Maîtriser la formule de dimension d'une couche convolutive.
- Visualiser l'effet du padding et du stride.
- Implémenter le max pooling et la moyenne, et comprendre leur rôle.

## Travail demandé

1. Répondre aux 3 questions de la section « À vous ».
2. Rendre le notebook exécuté.

## Mode d'emploi

- Exécuter les cellules **dans l'ordre** (`Shift + Entrée`).
- Les cellules marquées **« À vous »** sont à compléter : ce sont elles qui sont évaluées.
- Toutes les figures s'affichent dans le notebook ; si ce n'est pas le cas, vérifier que la
  première cellule a bien été exécutée.
- Reproductibilité : les graines aléatoires sont fixées. Deux exécutions donnent le même résultat.


In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(4112)

## 1. Padding, stride, pooling : trois réglages qui changent tout

On travaille sur de petits tenseurs pour voir exactement ce qui se passe.

In [2]:
I = np.arange(1, 26).reshape(5, 5).astype(float)
print(I)

[[ 1.  2.  3.  4.  5.]
 [ 6.  7.  8.  9. 10.]
 [11. 12. 13. 14. 15.]
 [16. 17. 18. 19. 20.]
 [21. 22. 23. 24. 25.]]


## 2. La formule de dimension

$$N_{out} = \left\lfloor \frac{N_{in} - K + 2P}{S} \right\rfloor + 1$$

$N_{in}$ : côté d'entrée · $K$ : côté du noyau · $P$ : padding · $S$ : stride.

In [3]:
def dim(N_in, K, P=0, S=1):
    return (N_in - K + 2 * P) // S + 1

for (N, K, P, S) in [(28, 3, 0, 1), (28, 3, 1, 1), (28, 5, 2, 1),
                     (28, 3, 0, 2), (32, 5, 0, 1), (224, 7, 3, 2)]:
    print(f"N_in={N:3d}  K={K}  P={P}  S={S}   ->   N_out={dim(N, K, P, S)}")

N_in= 28  K=3  P=0  S=1   ->   N_out=26
N_in= 28  K=3  P=1  S=1   ->   N_out=28
N_in= 28  K=5  P=2  S=1   ->   N_out=28
N_in= 28  K=3  P=0  S=2   ->   N_out=13
N_in= 32  K=5  P=0  S=1   ->   N_out=28
N_in=224  K=7  P=3  S=2   ->   N_out=112


## 3. Padding et stride, en image

Le padding ajoute des zéros autour de l'entrée ; le stride décide du pas de balayage.

In [4]:
def avec_padding(M, P):
    return np.pad(M, P, constant_values=0)

def correlation(M, K, S=1):
    kh, kw = K.shape
    h = (M.shape[0] - kh) // S + 1
    w = (M.shape[1] - kw) // S + 1
    out = np.zeros((h, w))
    for i in range(h):
        for j in range(w):
            out[i, j] = (M[i*S:i*S+kh, j*S:j*S+kw] * K).sum()
    return out

K = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], float)
print("valid  (P=0) :", correlation(I, K).shape)
print("same   (P=1) :", correlation(avec_padding(I, 1), K).shape)
print("stride 2     :", correlation(I, K, S=2).shape)

valid  (P=0) : (3, 3)
same   (P=1) : (5, 5)
stride 2     : (2, 2)


## 4. Pooling

Le pooling réduit la résolution sans rien apprendre. C'est l'opération la plus simple
du réseau — et l'une des plus efficaces.

In [5]:
def pool(M, mode="max", k=2, s=2):
    n = (M.shape[0] - k) // s + 1
    out = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            blk = M[i*s:i*s+k, j*s:j*s+k]
            out[i, j] = blk.max() if mode == "max" else blk.mean()
    return out

M = np.array([[1, 2, 5, 3],
              [4, 9, 2, 1],
              [7, 3, 8, 6],
              [2, 1, 4, 5]], float)
print("entree  :\n", M)
print("\nmax pooling 2x2 :\n", pool(M, "max"))
print("\nmoyenne 2x2 :\n", pool(M, "avg"))

entree  :
 [[1. 2. 5. 3.]
 [4. 9. 2. 1.]
 [7. 3. 8. 6.]
 [2. 1. 4. 5.]]

max pooling 2x2 :
 [[9. 5.]
 [7. 8.]]

moyenne 2x2 :
 [[4.   2.75]
 [3.25 5.75]]


## 5. À vous

**Question 1.** Quelle combinaison $(K, P, S)$ donne une sortie de même taille que l'entrée
$32 \times 32$ avec un noyau $5 \times 5$ ?

**Question 2.** Pourquoi le max pooling rend-il la détection moins sensible à un léger
décalage du motif ? Vérifiez en décalant d'un pixel la matrice $M$ ci-dessus.

**Question 3.** Combien de paramètres apprend une couche de pooling 3×3 ?

In [6]:
# ---- À VOUS ----
# 1) cherchez (K, P, S) tel que dim(32, K, P, S) == 32
# 2)

# 3) reponse :

## 6. Synthèse

| Réglage | Effet |
|---|---|
| **Padding** $P$ | contrôle la taille de sortie ; `same` la conserve |
| **Stride** $S$ | réduit la résolution d'un facteur $S$ |
| **Pooling** | réduit la résolution, **sans paramètre appris** |
| **Formule** | $N_{out} = \lfloor (N_{in} - K + 2P)/S \rfloor + 1$ |